# Notebook OOP-1 — Blueprints, alarms & coffee — instances in plain English

| | |
|---|---|
| **Focus** | Object-oriented Python (Beginner) |
| **Daily-life theme** | recipe cards, alarm clocks, shopping carts |
| **Pattern** | Story → Concept → Demo → **Exercise** → Solution |

**Prerequisites:** Basic Python syntax (`exercises/01–03`).

**Next notebook:** `02-oop-intermediate-house-rules.ipynb`

---

## Learning objectives

- Explain **class vs instance** using everyday metaphors.
- Initialize objects with `__init__` and store **state** on `self`.
- Call **methods** that change or read that state.

## Table of contents

1. **Recipe blueprint vs one tray of cookies**
2. **`AlarmClock` — ringing behavior**
3. **`CoffeeMaker` — simple actions**
4. ****Progressive drills** — water bottle → laundry → parking meter**
5. **Exercise — contacts list like your phone**

---

> **Tip:** When an analogy clicks, rename classes to AI-flavored names (`Embedder`, `ToolRegistry`) — same mechanics.

---


## 1 · Class = blueprint, instance = one real thing

**Daily life:** Architectural drawings describe *every* house on a street, but **your** house is one physical building with its own paint color.

**In Python:** `class CookieTray:` describes trays in general; `tray_a = CookieTray()` is **one** tray with its own cookie count.


In [ ]:
class CookieTray:
    """Blueprint for trays leaving the oven."""

    def __init__(self, flavor: str, count: int) -> None:
        self.flavor = flavor
        self.count = count

    def eat_one(self) -> None:
        if self.count > 0:
            self.count -= 1

    def describe(self) -> str:
        return f"{self.count}× {self.flavor} cookies"


birthday = CookieTray("chocolate", 12)
birthday.eat_one()
print(birthday.describe())


## 2 · `AlarmClock` — attributes + behavior together

**Daily life:** Two bedside alarms can show **different times** but **both know how to snooze**.

**Programming idea:** Data (`hour`, `minute`) lives next to actions (`snooze`).


In [ ]:
class AlarmClock:
    def __init__(self, label: str, hour: int = 7, minute: int = 0) -> None:
        self.label = label
        self.hour = hour
        self.minute = minute

    def display(self) -> str:
        return f"[{self.label}] {self.hour:02d}:{self.minute:02d}"

    def snooze(self, minutes: int = 9) -> None:
        self.minute += minutes
        while self.minute >= 60:
            self.minute -= 60
            self.hour += 1


work = AlarmClock("Weekdays", 6, 55)
work.snooze()
print(work.display())


## 3 · `CoffeeMaker` — methods encode habits

**Daily life:** Push-button steps (`grind`, `brew`) reuse the same machine state (`beans_left`).


In [ ]:
class CoffeeMaker:
    def __init__(self, beans_grams: float) -> None:
        self.beans_grams = beans_grams

    def grind(self, grams: float) -> None:
        self.beans_grams = max(0.0, self.beans_grams - grams)

    def can_brew(self, shot_grams: float = 18.0) -> bool:
        return self.beans_grams >= shot_grams


kitchen = CoffeeMaker(40)
kitchen.grind(10)
print("Enough for espresso?", kitchen.can_brew())


---

## Progressive examples — **easy → harder**

Standalone cells below ramp difficulty: **one number + one verb** → **small collections** → **rules + guards**. Compare each class's **state** (`self....`) before and after methods run.


### A · Easiest — **water bottle** (one dial: milliliters left)

**Daily life:** Each sip drains liquid; you can't magically sip below zero.


In [ ]:
class WaterBottle:
    def __init__(self, milliliters: float) -> None:
        self.ml = milliliters

    def sip(self, amount: float) -> float:
        """Return how much was actually swallowed."""
        swallowed = min(amount, self.ml)
        self.ml -= swallowed
        return swallowed


bottle = WaterBottle(500)
print("sip", bottle.sip(120), "left", bottle.ml)
print("sip huge", bottle.sip(9999), "left", bottle.ml)


### B · Easy — **laundry basket** (hidden list)

**Daily life:** Toss socks in, grab something to wash—internal pile grows/shrinks without exposing raw chaos.


In [ ]:
class LaundryBasket:
    def __init__(self) -> None:
        self._pile: list[str] = []

    def toss(self, garment: str) -> None:
        self._pile.append(garment)

    def grab_any(self) -> str | None:
        return self._pile.pop() if self._pile else None

    def load_size(self) -> int:
        return len(self._pile)


hamper = LaundryBasket()
hamper.toss("sock")
hamper.toss("tee")
print("size", hamper.load_size(), "grabbed", hamper.grab_any(), "remaining", hamper.load_size())


### C · Medium — **parking meter** (accumulate then burn time)

**Daily life:** Coins buy minutes; driving away consumes minutes until expired.


In [ ]:
class ParkingMeter:
    def __init__(self) -> None:
        self.minutes_left = 0

    def add_quarters(self, quarters: int) -> None:
        if quarters < 0:
            raise ValueError("no negative quarters")
        self.minutes_left += quarters * 15  # toy rule: 15 min each

    def drive_minutes(self, minutes: int) -> None:
        self.minutes_left = max(0, self.minutes_left - minutes)

    def is_expired(self) -> bool:
        return self.minutes_left == 0


m = ParkingMeter()
m.add_quarters(2)
print("after coins", m.minutes_left)
m.drive_minutes(40)
print("after commute", m.minutes_left, "expired?", m.is_expired())


### Exercise — Phone contacts (beginner)

**Daily life:** Your contacts app stores names → numbers and lets you search.

Implement `ContactBook`:

- `add(name, phone)` — store contact (`phone` string ok).
- `find(prefix)` — return **sorted list** of names that **start with** `prefix` (case-sensitive is fine).

Starter asserts included.


In [ ]:
class ContactBook:
    def __init__(self) -> None:
        raise NotImplementedError

    def add(self, name: str, phone: str) -> None:
        raise NotImplementedError

    def find(self, prefix: str) -> list[str]:
        raise NotImplementedError


cb = ContactBook()
cb.add("Sam", "555-0101")
cb.add("Sara", "555-0102")
cb.add("Mo", "555-0199")
assert set(cb.find("Sa")) == {"Sam", "Sara"}
print("Exercise OK")


### Solution — ContactBook

<details>
<summary>Reveal solution</summary>

```python
class ContactBook:
    def __init__(self) -> None:
        self._contacts: dict[str, str] = {}

    def add(self, name: str, phone: str) -> None:
        self._contacts[name] = phone

    def find(self, prefix: str) -> list[str]:
        hits = [n for n in self._contacts if n.startswith(prefix)]
        return sorted(hits)
```

</details>
